In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import warnings
warnings.filterwarnings('ignore')
import seaborn as sns

%pip install kagglehub catboost xgboost tqdm -q

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

from sklearn.metrics import accuracy_score, f1_score

from catboost import CatBoostClassifier

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

q3 = os.path.join(path, 'Q3_data.csv')
print("Path to dataset files:", path)


In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(q3)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df.shape

In [ ]:
# Task 1: Write your code here:

In [ ]:
df = df.drop(columns='D_142')
# not a error but bcs a ran it again, i dropped this column bcs it had soo many missing values and its not needed

In [ ]:

columns = [col for col in df.columns]

In [ ]:
for col in columns:
  df[col] = df[col].fillna(df[col].median())


In [ ]:
df.isnull().sum()

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

In [ ]:
# Task 3: Write your code here:
df.select_dtypes(include=['object']).columns

In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()
X = scaler.fit_transform(df.drop(columns='Target'))
y = df['Target']

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
'Already did that :)'
#X = scaler.fit_transform(df.drop(columns='Target'))
#y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4)

accuracy_scores = []
f1_scores = []


for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  accuracy_scores.append(accuracy_score(y_test, y_pred))
  f1_scores.append(f1_score(y_test, y_pred, average='macro'))



print(f"  Accuracy:  {np.mean(accuracy_scores):.4f}")
print(f"  F1-Score:  {np.mean(f1_scores):.4f}")



In [ ]:
feature_cols = df.columns.drop('Target')


In [ ]:
feature_cols

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(20)



plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(f"THIS IS THE GOLDEN RETRIEVER I MEAN FEATURE [P_2] and those are his values {df['P_2']}")
golden_feature = df['P_2']

In [ ]:
# Task Bonus: Write your code here:

X_new = df.select_dtypes(include=['P_2']).columns


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4)

accuracy_scores = []
f1_scores = []


for fold_idx, (train_index, test_index) in enumerate(skf.split(X_new, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X_new[train_index], X_new[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)
  y_pred = model.predict(X_new)

  accuracy_scores.append(accuracy_score(y_test, y_pred))
  f1_scores.append(f1_score(y_test, y_pred, average='macro'))



print(f"  Accuracy:  {np.mean(accuracy_scores):.4f}")
print(f"  F1-Score:  {np.mean(f1_scores):.4f}")
